![](data/flopylogo_sm.png)

# 01: Introduction to FloPy

The second part of this course focuses on [FloPy](https://github.com/modflowpy/flopy), a Python package for creating, running, and post-processing MODFLOW-based groundwater flow and transport models. Why would we want this? MODFLOW—especially older versions—has idiosyncratic input and output that can be difficult to work with directly. FloPy translates MODFLOW input and output into the general Python data structures we explored in the first part of the course, making it easier to script groundwater modeling workflows with the entire scientific Python ecosysem.

 FloPy was originally developed by [Mark Bakker and others (2016)](https://ngwa.onlinelibrary.wiley.com/doi/10.1111/gwat.12413) for working with MODFLOW 2005 (including MODFLOW-NWT) and earlier versions of MODFLOW. [FloPy has since been expanded](https://doi.org/10.1111/gwat.13327) to support [MODFLOW 6](https://github.com/MODFLOW-USGS/modflow6), the current version of MODFLOW that includes general support for structured and unstructured grids, tight coupling of multiple models within a simulation, and a redesigned input structure that is meant to be more intuitive and human readable. FloPy support for MODFLOW 6 is tightly coupled to the MODFLOW 6 code, in that the relevant FloPy code is auto-generated from text files (_definition files_) that describe MODFLOW 6 models and packages. As a result, MODFLOW 2005-style and MODFLOW 6 functionality within FloPy is accessed through different subpackages—`flopy.modflow` and `flopy.mf6` respectively. 

For example, to instantiate a MODFLOW-NWT model instance, one would enter:


In [ ]:
import flopy

model = flopy.modflow.Modflow(modelname='test', version='mfnwt')  # or 'mf2005' for MODFLOW 2005 (the default)

For MODFLOW 6, we first have to create a simulation, which we then assign the model to:

In [ ]:
sim = flopy.mf6.MFSimulation(sim_name='test')
gwf = flopy.mf6.ModflowGwf(sim, modelname='test')

As you can see, in addition to the different FloPy subpackages, the syntax is different between the two versions. In general, the syntax for the `flopy.modflow` and `flopy.mf6` subpackages follows the respective MODFLOW versions they support. For example, `ModflowGwf` is simply a [CamelCase](https://en.wikipedia.org/wiki/Camel_case) representation of the MODFLOW 6 Groundwater Flow (GWF) model. Similarly, the Discretization Package (for the GWF model) class is named `ModflowGwfdis`. Arguments to the `ModflowGwfdis` constructor follow the input variables to MODFLOW 6.

In [ ]:
dis = flopy.mf6.ModflowGwfdis(
    gwf,
    nlay=3,
    nrow=21,
    ncol=20,
    delr=500.,
    delc=500.,
    top=400.0,
    botm=[220, 200, 0],
)

The [MODFLOW 6 Input and Output Guide](https://modflow6.readthedocs.io/en/latest/mf6io.html) ([individual releases](https://github.com/MODFLOW-USGS/modflow6/releases) also contain a PDF version) can therefore be a valuable resource for understanding how to use Flopy.

Other Flopy subpackages (`flopy.modpath`, `flopy.mt3d`, etc) provide varying levels of support for previous versions of MODFLOW or related software such as MODPATH and MT3D; or general (Flopy-wide) support for discretization, exporting or other processing (`flopy.discretization`, `flopy.export`, `flopy.utils`, etc.).

This class will focus almost exclusively on FloPy support for MODFLOW 6. Examples and documentation for using FloPy in other contexts is available in the [FloPy online documentation](https://flopy.readthedocs.io/en/stable/index.html). The [Examples gallery](https://flopy.readthedocs.io/en/stable/examples.html) are a resource that can be used to quickly learn the underlying capabilities of FloPy.

---

## 🤖 AI Sidebar: Generating MODFLOW 6 Boilerplate with AI

*This sidebar demonstrates how an AI coding assistant handles FloPy/MODFLOW 6 code generation — and where it commonly goes wrong. All AI-generated code below was captured ahead of time; no live AI calls are made.*

**Time estimate:** ~5-8 minutes

### The Task

In this notebook, you learned how FloPy creates MODFLOW 6 simulations: first a `MFSimulation`, then a `ModflowGwf` model, then packages like `ModflowGwfdis` for discretization. The naming convention follows MODFLOW 6 input — `ModflowGwf` + package abbreviation (e.g., `dis`, `npf`, `ic`).

Let's see what happens when we ask an AI assistant to generate the boilerplate for a complete MODFLOW 6 simulation. This is a common use case — setting up the simulation scaffolding is repetitive, and AI should be able to help. But FloPy's dual API (`flopy.modflow` vs. `flopy.mf6`) is a frequent source of AI confusion.

### AI Approach

**Prompt used:**

> *"Write Python code using FloPy to create a simple MODFLOW 6 groundwater flow simulation. The model should have 1 layer, 10 rows, 10 columns, with 100m cell spacing. Set constant head boundaries on the left (h=10m) and right (h=5m) sides. Use a hydraulic conductivity of 1e-4 m/s. Include all required packages to run the simulation."*

**AI-generated code (first attempt — contains errors):**

In [ ]:
# AI-GENERATED CODE (pre-captured) — CONTAINS ERRORS, DO NOT RUN
# This code is shown for educational purposes to highlight common AI mistakes.

ai_code_attempt_1 = """
import flopy

# Create the model
model = flopy.modflow.Modflow('simple_model', exe_name='mf6')  # ERROR 1: wrong API!

# Discretization
dis = flopy.modflow.ModflowDis(                                 # ERROR 2: MF2005 package!
    model, nlay=1, nrow=10, ncol=10,
    delr=100.0, delc=100.0,
    top=10.0, botm=0.0
)

# Hydraulic conductivity
lpf = flopy.modflow.ModflowLpf(model, hk=1e-4)                 # ERROR 3: LPF is MF2005!

# Constant head boundaries
chd_data = []
for row in range(10):
    chd_data.append([0, row, 0, 10.0, 10.0])      # left side
    chd_data.append([0, row, 9, 5.0, 5.0])         # right side
chd = flopy.modflow.ModflowChd(model, stress_period_data=chd_data)  # ERROR 4: MF2005 CHD!

# Solver
pcg = flopy.modflow.ModflowPcg(model)               # ERROR 5: PCG is MF2005 solver!

# Output control
oc = flopy.modflow.ModflowOc(model)                  # ERROR 6: MF2005 OC!

# Write and run
model.write_input()
model.run_model()
"""

print("See the errors annotated in the code above.")
print("The AI used flopy.modflow (MODFLOW 2005) instead of flopy.mf6 (MODFLOW 6)!")

### Common AI Errors with FloPy

The code above demonstrates the **most common AI mistake** with FloPy: confusing the `flopy.modflow` (MODFLOW 2005) API with the `flopy.mf6` (MODFLOW 6) API. Here's a breakdown:

| AI Wrote (Wrong ❌) | Correct for MF6 (✅) | Why AI Gets Confused |
|---------------------|----------------------|---------------------|
| `flopy.modflow.Modflow(...)` | `flopy.mf6.MFSimulation(...)` + `flopy.mf6.ModflowGwf(...)` | MF6 requires a simulation *and* a model |
| `flopy.modflow.ModflowDis(...)` | `flopy.mf6.ModflowGwfdis(...)` | MF6 class names include the model type (`Gwf`) |
| `flopy.modflow.ModflowLpf(...)` | `flopy.mf6.ModflowGwfnpf(...)` | LPF was replaced by NPF in MF6 |
| `flopy.modflow.ModflowChd(...)` | `flopy.mf6.ModflowGwfchd(...)` | Same concept, different class and data format |
| `flopy.modflow.ModflowPcg(...)` | `flopy.mf6.ModflowIms(...)` | PCG replaced by IMS in MF6 |
| `model.write_input()` | `sim.write_simulation()` | MF6 writes at the simulation level |

**Why does this happen?** AI models are trained on large amounts of code, and there is far more `flopy.modflow` code on the internet (it's been around since 2016) than `flopy.mf6` code. The AI defaults to the more common pattern even when you explicitly ask for MODFLOW 6.

Other common AI errors with FloPy include:
- **Incorrect package naming:** Writing `ModflowGwfDis` (capital D) instead of `ModflowGwfdis` (lowercase)
- **Deprecated API patterns:** Using older FloPy function signatures that have been updated
- **Missing required packages:** Forgetting TDIS (time discretization) or IMS (solver), which are required for MF6
- **Wrong stress period data format:** MF6 uses a different format for stress period data than MF2005

### Corrected Code

Here is the correct MODFLOW 6 version using `flopy.mf6`. Compare this with the AI's first attempt above:

In [ ]:
import flopy
import numpy as np
import os

# --- Corrected MODFLOW 6 code ---

# MF6 requires a simulation object first (not just a model)
workspace = os.path.join('temp', 'ai_sidebar_mf6')
sim = flopy.mf6.MFSimulation(
    sim_name='simple_mf6',
    sim_ws=workspace
)

# Time discretization — required for MF6, often forgotten by AI
tdis = flopy.mf6.ModflowTdis(
    sim,
    nper=1,
    perioddata=[(1.0, 1, 1.0)]  # (perlen, nstp, tsmult)
)

# Iterative Model Solution — MF6 uses IMS, not PCG
ims = flopy.mf6.ModflowIms(
    sim,
    complexity='simple'
)

# Groundwater Flow model — note: assigned to the simulation
gwf = flopy.mf6.ModflowGwf(
    sim,
    modelname='simple_mf6',
    save_flows=True
)

# Discretization — note the class name: ModflowGwfdis (not ModflowDis)
dis = flopy.mf6.ModflowGwfdis(
    gwf,
    nlay=1, nrow=10, ncol=10,
    delr=100.0, delc=100.0,
    top=10.0, botm=[0.0]
)

# Node Property Flow — MF6 uses NPF, not LPF
npf = flopy.mf6.ModflowGwfnpf(
    gwf,
    k=1e-4  # hydraulic conductivity in m/s
)

# Initial conditions
ic = flopy.mf6.ModflowGwfic(
    gwf,
    strt=10.0
)

# Constant head boundaries — MF6 format uses [(layer, row, col), head]
chd_data = []
for row in range(10):
    chd_data.append([(0, row, 0), 10.0])    # left side: h = 10 m
    chd_data.append([(0, row, 9), 5.0])     # right side: h = 5 m

chd = flopy.mf6.ModflowGwfchd(
    gwf,
    stress_period_data=chd_data
)

# Output control
oc = flopy.mf6.ModflowGwfoc(
    gwf,
    head_filerecord='simple_mf6.hds',
    saverecord=[('HEAD', 'ALL')]
)

# Write simulation files (not model.write_input()!)
sim.write_simulation()

print("Simulation files written to:", workspace)
print("Packages created:", [pkg.name[0] for pkg in gwf.packagelist])

### Identifying Errors Using FloPy Docs and Error Messages

When AI-generated FloPy code doesn't work, here's how to diagnose the problem:

**1. Check the error message carefully.** FloPy error messages often tell you exactly what's wrong:

```python
# If you try to use flopy.modflow classes with an MF6 simulation, you'll get:
# AttributeError: module 'flopy.modflow' has no attribute 'ModflowGwfdis'
#
# This tells you: ModflowGwfdis lives in flopy.mf6, not flopy.modflow
```

**2. Use the FloPy class naming convention.** For MODFLOW 6, all package classes follow the pattern:
- `Modflow` + model type (`Gwf`, `Gwt`) + package abbreviation (lowercase)
- Examples: `ModflowGwfdis`, `ModflowGwfnpf`, `ModflowGwfchd`, `ModflowGwfoc`
- Simulation-level packages skip the model type: `ModflowTdis`, `ModflowIms`

**3. Consult the FloPy docs.** The [FloPy documentation](https://flopy.readthedocs.io/en/stable/) and the [MODFLOW 6 Input and Output Guide](https://modflow6.readthedocs.io/en/latest/mf6io.html) are the authoritative references. When AI code uses a class name you don't recognize, search the docs.

**4. Use Python's `dir()` and `help()` to explore:**

In [ ]:
# List all MF6 GWF package classes — useful for checking AI-generated class names
gwf_classes = [name for name in dir(flopy.mf6) if name.startswith('ModflowGwf')]
print("Available ModflowGwf* classes (first 15):")
for cls in gwf_classes[:15]:
    print(f"  flopy.mf6.{cls}")

In [ ]:
# Check the signature of a specific class to verify AI-generated arguments
help(flopy.mf6.ModflowGwfnpf.__init__)

### Comparison

| Aspect | Manual Approach | AI Approach |
|--------|----------------|-------------|
| **Time** | 10-15 min following docs/examples | ~1 min to generate, but 5-10 min to fix errors |
| **Understanding** | You learn the MF6 simulation structure as you build it | You may get working code without understanding the sim → model → package hierarchy |
| **Correctness** | Following the docs ensures correct API usage | High risk of `flopy.modflow` vs `flopy.mf6` confusion |
| **Completeness** | You know which packages are required (TDIS, IMS, DIS, etc.) | AI often forgets TDIS or IMS, which are required for MF6 |

### What Could Go Wrong

The most dangerous AI errors with FloPy are **silent failures** — code that runs without crashing but builds the wrong kind of model:

1. **API confusion (`flopy.modflow` vs `flopy.mf6`):** The AI may generate syntactically valid MODFLOW 2005 code when you asked for MODFLOW 6. The code runs, but you're using the wrong version of MODFLOW entirely.

2. **Incorrect package naming:** Writing `ModflowGwfDis` (capital D) instead of `ModflowGwfdis` raises an `AttributeError` — but the error message can be confusing if you don't know the naming convention.

3. **Wrong stress period data format:** MF2005 CHD uses `[lay, row, col, shead, ehead]` while MF6 CHD uses `[(lay, row, col), head]`. Mixing these up causes cryptic errors or wrong boundary conditions.

4. **Missing required packages:** MF6 simulations require TDIS and IMS at the simulation level. AI often forgets these because MF2005 handles time and solving differently.

### When to Use AI Here

- ✅ **Well-suited for AI:** Generating the repetitive boilerplate structure (sim → model → packages) once you know the correct pattern
- ✅ **Well-suited for AI:** Looking up parameter names and default values for packages you've used before
- ✅ **Well-suited for AI:** Generating stress period data arrays for simple boundary conditions
- ⚠️ **Requires manual understanding:** Knowing whether to use `flopy.modflow` or `flopy.mf6` for your version of MODFLOW
- ⚠️ **Requires manual understanding:** Verifying that the correct packages are included (TDIS, IMS, DIS, NPF, IC, OC)
- ⚠️ **Requires manual understanding:** Checking that stress period data format matches the MF6 specification
- ⚠️ **Requires manual understanding:** Confirming that model results are physically reasonable (heads within expected bounds)

---

### 🤖 AI Exercise: Build a Confined Aquifer Model from a Description

In this exercise, you'll practice the AI-assisted workflow for constructing a MODFLOW 6 model. You'll describe a hydrogeologic scenario in plain language, use an AI assistant to generate the FloPy code, and then verify that the model runs and produces physically reasonable results.

**Scenario:**

> A confined aquifer is 50 meters thick, with a horizontal extent of 1000m × 1000m. The aquifer has a hydraulic conductivity of 10 m/day. The left boundary has a constant head of 100m and the right boundary has a constant head of 95m. There is no recharge (the aquifer is confined). Build a steady-state MODFLOW 6 model with 1 layer, 10 rows, and 10 columns.

**Expected results (ground truth):**
- Heads should decrease linearly from 100m (left) to 95m (right)
- All heads should be between 95m and 100m
- The head gradient should be uniform: (100 - 95) / 1000 = 0.005 m/m

**Step 1:** Write your prompt in the cell below. Be specific about which version of MODFLOW and which FloPy subpackage to use — this is the key to avoiding the `flopy.modflow` vs `flopy.mf6` confusion.

In [ ]:
# Write your prompt here as a comment or string:
my_prompt = """
YOUR PROMPT HERE

Tip: explicitly mention "flopy.mf6" and "MODFLOW 6" in your prompt.
Tip: specify the required packages: TDIS, IMS, GWF, DIS, NPF, IC, CHD, OC.
"""

**Step 2:** Paste or write the AI-generated code in the cell below. If the AI uses `flopy.modflow` instead of `flopy.mf6`, that's a learning opportunity — correct it using the comparison table above.

In [ ]:
# Paste your AI-generated MODFLOW 6 model code here:



**Step 3:** Verify the model results. If your model ran successfully and wrote output, use the cells below to check that heads are physically reasonable.

In [ ]:
# Verification — uncomment and adapt once your model has run
# (Replace 'your_sim' and 'your_model_name' with your variable names)

# # Load the head results
# head_file = your_sim.simulation_data.mfpath.get_sim_path() / 'your_model_name.hds'
# hds = flopy.utils.HeadFile(str(head_file))
# heads = hds.get_data()
#
# print(f"Head range: {heads.min():.2f} to {heads.max():.2f} m")
# print(f"Expected range: 95.0 to 100.0 m")
# print()
# print(f"Heads physically reasonable: {95.0 <= heads.min() and heads.max() <= 100.0}")
# print()
# # Check that heads decrease from left to right
# mid_row = heads[0, 5, :]  # middle row, all columns
# print(f"Heads along middle row (left to right):")
# print(f"  {np.array2string(mid_row, precision=2)}")
# print(f"  Monotonically decreasing: {all(mid_row[i] >= mid_row[i+1] for i in range(len(mid_row)-1))}")

### ✅ Verification Checklist

- [ ] Code uses `flopy.mf6` (not `flopy.modflow`)
- [ ] All required MF6 packages are present: TDIS, IMS, DIS, NPF, IC, CHD, OC
- [ ] Simulation files write without errors
- [ ] Model runs successfully (if MODFLOW 6 executable is available)
- [ ] Heads are between 95m and 100m (physically reasonable for this scenario)
- [ ] Heads decrease from left (100m) to right (95m)
- [ ] Code quality: no deprecated API patterns, correct class naming convention

### Prompt Log

In [ ]:
# Prompt Log
# ----------
# Prompt version 1: (paste your first prompt here)
#
# Result: (did the AI use flopy.mf6? were all packages included?)
#
# Prompt version 2 (if needed): (what did you change to fix the API confusion?)
#
# Result: (did the model write/run successfully?)
#
# Number of iterations to reach correct result: 
#
# Key lesson about prompting for FloPy code: